# Работен процес с човешка намеса с Microsoft Agent Framework

## 🎯 Учебни цели

В този бележник ще научите как да прилагате работни процеси с **човек в цикъла** с помощта на `RequestInfoExecutor` от Microsoft Agent Framework. Този мощен модел ви позволява да поставите на пауза AI работни процеси, за да съберете човешки вход, правейки вашите агенти интерактивни и давайки възможност на хората да контролират критични решения.

## 🔄 Какво е Човек в цикъла?

**Човек в цикъла (HITL)** е модел на проектиране, при който AI агентите спират изпълнението си, за да поискат човешки вход преди да продължат. Това е от съществено значение за:

- ✅ **Критични решения** - Получаване на човешко одобрение преди предприемане на важни действия
- ✅ **Неясни ситуации** - Позволяване на хората да изяснят, когато AI е несигурен
- ✅ **Предпочитания на потребителите** - Приканване на потребителите да избират между няколко опции
- ✅ **Съответствие и безопасност** - Осигуряване на човешки контрол при регулирани операции
- ✅ **Интерактивни преживявания** - Създаване на разговорни агенти, които отговарят на потребителски вход

## 🏗️ Как работи в Microsoft Agent Framework

Фреймуъркът предоставя три ключови компонента за HITL:

1. **`RequestInfoExecutor`** - Специален изпълнител, който поставя на пауза работния процес и излъчва `RequestInfoEvent`
2. **`RequestInfoMessage`** - Базов клас за типизирани заявки, изпратени до хората
3. **`RequestResponse`** - Свързва човешките отговори с оригиналните заявки чрез `request_id`

**Шаблон на работния процес:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Нашият пример: Резервация на хотел с потребителско потвърждение

Ще надградим условния работен процес като добавим човешко потвърждение **преди** да предложим алтернативни дестинации:

1. Потребителят иска дестинация (напр. "Париж")
2. `availability_agent` проверява дали има налични стаи
3. **Ако няма стаи** → `confirmation_agent` пита "Искате ли да видите алтернативи?"
4. Работният процес се **поставя на пауза** чрез `RequestInfoExecutor`
5. **Човекът отговаря** с "да" или "не" чрез конзолен вход
6. `decision_manager` насочва на базата на отговора:
   - **Да** → Показва алтернативни дестинации
   - **Не** → Отказва заявката за резервация
7. Показва крайния резултат

Това демонстрира как да дадете възможност на потребителите да контролират предложенията на агента!

---

Нека започваме! 🚀


## Стъпка 1: Импортиране на необходимите библиотеки

Импортираме стандартните компоненти на Agent Framework плюс **класове специфични за човека в цикъла**:
- `RequestInfoExecutor` - Изпълнител, който паузира работния процес за човешка намеса
- `RequestInfoEvent` - Събитие, генерирано когато се иска човешка намеса
- `RequestInfoMessage` - Базов клас за типизирани заявки
- `RequestResponse` - Свързва човешките отговори със заявките
- `WorkflowOutputEvent` - Събитие за откриване на изходи от работния процес


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Стъпка 2: Дефиниране на Pydantic модели за структурирани изходи

Тези модели дефинират **схемата**, която агентите ще връщат. Запазваме всички модели от условния работен процес и добавяме:

**Нови за Човек-в-циклa:**
- `HumanFeedbackRequest` - Подклас на `RequestInfoMessage`, който дефинира полезния товар на заявката, изпращан към хората
  - Съдържа `prompt` (въпрос за задаване) и `destination` (контекст за недостъпния град)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Стъпка 3: Създаване на инструмента за резервация на хотел

Същият инструмент от условния работен процес - проверява дали има свободни стаи на мястото за настаняване.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## Стъпка 4: Дефиниране на условни функции за маршрутизация

Имаме нужда от **четири условни функции** за нашия работен процес с човек в цикъла:

**От условен работен процес:**
1. `has_availability_condition` - Маршрутизира, когато хотелите СА налични
2. `no_availability_condition` - Маршрутизира, когато хотелите НЕ са налични

**Нови за човек в цикъла:**
3. `user_wants_alternatives_condition` - Маршрутизира, когато потребителят казва "да" на алтернативи
4. `user_declines_alternatives_condition` - Маршрутизира, когато потребителят казва "не" на алтернативи


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Стъпка 5: Създайте изпълнител на Decision Manager

Това е **ядрото на патърна човек-в-цикъла**! `DecisionManager` е персонализиран `Executor`, който:

1. **Получава човешка обратна връзка** чрез обекти `RequestResponse`
2. **Обработва решението на потребителя** (да/не)
3. **Рутира работния процес**, като изпраща съобщения към подходящи агенти

Ключови характеристики:
- Използва декоратора `@handler`, за да излага методи като стъпки в работния поток
- Получава `RequestResponse[HumanFeedbackRequest, str]`, съдържащ както оригиналната заявка, така и отговора на потребителя
- Предава прости съобщения "да" или "не", които задействат нашите условни функции


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## Стъпка 6: Създаване на потребителски изпълнител за визуализация

Същият изпълнител за визуализация от условния работен поток - връща крайни резултати като изход на работния поток.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Стъпка 7: Зареждане на променливи на средата

Конфигурирайте клиента на LLM (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## Стъпка 8: Създаване на AI агенти и изпълнители

Създаваме **шест компонента за работния процес**:

**Агенти (обвити в AgentExecutor):**
1. **availability_agent** - Проверява наличността на хотела чрез инструмента
2. **confirmation_agent** - 🆕 Подготвя заявката за потвърждение от човек
3. **alternative_agent** - Предлага алтернативни градове (когато потребителят каже да)
4. **booking_agent** - Подкрепя резервацията (когато има налични стаи)
5. **cancellation_agent** - 🆕 Обработва съобщението за анулиране (когато потребителят каже не)

**Специални изпълнители:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor`, който спира работния процес за човешки вход
7. **decision_manager** - 🆕 Персонализиран изпълнител, който насочва в зависимост от човешкия отговор (вече е дефиниран по-горе)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Стъпка 9: Създаване на работния процес с човек в цикъла

Сега ще конструираме графа на работния процес с **условно маршрутизиране**, включително пътя с човек в цикъла:

**Структура на работния процес:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Ключови връзки:**
- `availability_agent → confirmation_agent` (когато няма стаи)
- `confirmation_agent → prepare_human_request` (преобразуване на тип)
- `prepare_human_request → request_info_executor` (пауза за човек)
- `request_info_executor → decision_manager` (винаги - осигурява RequestResponse)
- `decision_manager → alternative_agent` (когато потребителят каже "да")
- `decision_manager → cancellation_agent` (когато потребителят каже "не")
- `availability_agent → booking_agent` (когато са налични стаи)
- Всички пътища завършват на `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Стъпка 10: Стартиране на тестов случай 1 - Град БЕЗ наличност (Париж с човешко потвърждение)

Този тест демонстрира **пълния цикъл с човешко участие**:

1. Заявка за хотел в Париж
2. availability_agent проверява → Няма стаи
3. confirmation_agent създава въпрос към човека
4. request_info_executor **пауза на работния процес** и излъчва `RequestInfoEvent`
5. **Приложението засича събитието и подканя потребителя в конзолата**
6. Потребителят пише "да" или "не"
7. Приложението изпраща отговора чрез `send_responses_streaming()`
8. decision_manager маршрутизира на базата на отговора
9. Показва се крайният резултат

**Основен модел:**
- Използвайте `workflow.run_stream()` за първата итерация
- Използвайте `workflow.send_responses_streaming(pending_responses)` за следващи итерации
- Слушайте за `RequestInfoEvent`, за да засечете кога е нужна човешка намеса
- Слушайте за `WorkflowOutputEvent`, за да уловите крайния резултат


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Стъпка 11: Стартиране на тестов случай 2 - Град С наличност (Стокхолм - Не е необходим човешки вход)

Този тест демонстрира **директния път**, когато има налични стаи:

1. Заявка за хотел в Стокхолм
2. availability_agent проверява → Налични стаи ✅
3. booking_agent предлага резервация
4. display_result показва потвърждение
5. **Не е необходим човешки вход!**

Работният процес напълно заобикаля пътя с човешко участие, когато има налични стаи.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Основни изводи и добри практики за човека в цикъла

### ✅ Какво научихте:

#### 1. **Шаблон RequestInfoExecutor**
Шаблонът човек в цикъла в Microsoft Agent Framework използва три ключови компонента:
- `RequestInfoExecutor` - Поставя на пауза работния процес и излъчва събития
- `RequestInfoMessage` - Основен клас за типизирани заявки (наследявайте този клас!)
- `RequestResponse` - Свързва човешките отговори с оригиналните заявки

**Критично разбиране:**
- `RequestInfoExecutor` НЕ събира входни данни сам - само поставя работния процес на пауза
- Вашият код на приложението трябва да слуша за `RequestInfoEvent` и да събира входни данни
- Трябва да извикате `send_responses_streaming()` с дикт, който свързва `request_id` с отговора на потребителя

#### 2. **Шаблон за изпълнение на потоци**
```python
# Първа итерация
stream = workflow.run_stream(initial_request)

# Последващи итерации (след човешки вход)
stream = workflow.send_responses_streaming(pending_responses)

# Винаги обработвайте събитията
events = [event async for event in stream]
```

#### 3. **Архитектура, базирана на събития**
Слушайте за специфични събития, за да контролирате работния процес:
- `RequestInfoEvent` - Нужен е човешки вход (работният процес е на пауза)
- `WorkflowOutputEvent` - Краен резултат е наличен (работният процес е завършен)
- `WorkflowStatusEvent` - Промени в състоянието (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS и др.)

#### 4. **Персонализирани изпълнители с @handler**
`DecisionManager` демонстрира как да създадете изпълнители, които:
- Използват декоратора `@handler` за излагане на методи като стъпки в работния процес
- Получават типизирани съобщения (напр. `RequestResponse[HumanFeedbackRequest, str]`)
- Насочват работния процес чрез изпращане на съобщения към други изпълнители
- Достъпват контекста чрез `WorkflowContext`

#### 5. **Условно маршрутизиране с човешки решения**
Можете да създавате условни функции, които оценяват човешките отговори:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Приложения в реалния свят:

1. **Одобрителни работни процеси**
   - Получаване на одобрение от мениджър преди обработка на отчетите за разходи
   - Изискване на човешка проверка преди изпращане на автоматизирани имейли
   - Потвърждение на транзакции с висока стойност преди изпълнение

2. **Модериране на съдържание**
   - Отбелязване на съмнително съдържание за човешка проверка
   - Помолете модератори да вземат окончателно решение в гранични случаи
   - Ескалация към хора, когато доверието на ИИ е ниско

3. **Обслужване на клиенти**
   - Оставете ИИ да обработва рутинни въпроси автоматично
   - Ескалирайте сложни проблеми към човешки агенти
   - Попитайте клиента дали иска да говори с човек

4. **Обработка на данни**
   - Помолете хора да разрешат неясни входове в данните
   - Потвърдете интерпретациите на ИИ за неясни документи
   - Позволете на потребителите да избират между няколко валидни интерпретации

5. **Критични за безопасността системи**
   - Изисква човешко потвърждение преди необратими действия
   - Получаване на одобрение преди достъп до чувствителни данни
   - Потвърждаване на решения в регулирани индустрии (здравеопазване, финанси)

6. **Интерактивни агенти**
   - Създавайте разговорни ботове, които задават последващи въпроси
   - Създавайте помощници, които насочват потребителите през сложни процеси
   - Проектирайте агенти, които стъпка по стъпка си сътрудничат с хора

### 🔄 Сравнение: С човек в цикъла и без

| Функция | Условен работен процес | Работен процес с човек в цикъла |
|---------|---------------------|---------------------------|
| **Изпълнение** | Еднократно `workflow.run()` | Цикъл с `run_stream()` + `send_responses_streaming()` |
| **Потребителски вход** | Няма (напълно автоматизиран) | Интерактивни заявки чрез `input()` или UI |
| **Компоненти** | Агенти + Изпълнители | + RequestInfoExecutor + DecisionManager |
| **Събития** | Само AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent и др. |
| **Пауза** | Без пауза | Работният процес спира при RequestInfoExecutor |
| **Човешки контрол** | Без човешки контрол | Хората взимат ключови решения |
| **Приложение** | Автоматизация | Сътрудничество и контрол |

### 🚀 Разширени шаблони:

#### Множество точки за човешко решение
Може да имате множество `RequestInfoExecutor` възли в един и същ работен процес:
```python
.add_edge(agent1, request_info_1)  # Първо човешко решение
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Второ човешко решение
.add_edge(decision_manager_2, final_agent)
```

#### Обработка на таймаути
Внедрете таймаути за човешки отговори:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # По подразбиране безопасен избор
```

#### Богата интеграция с потребителски интерфейс
Вместо `input()`, интегрирайте с уеб UI, Slack, Teams и др.:
```python
if isinstance(event, RequestInfoEvent):
    # Изпратете известие до предпочитания канал на потребителя
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Условен човек в цикъла
Питайте за човешки вход само в конкретни ситуации:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Изпращайте към човек само ако увереността е ниска или стойността е висока
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Най-добри практики:

1. **Винаги наследявайте RequestInfoMessage**
   - Осигурява типова безопасност и валидиране
   - Позволява богата контекстова информация за визуализация на UI
   - Изяснява намерението на всеки тип заявка

2. **Използвайте описателни подкани**
   - Включвайте контекст за това, което питате
   - Обяснявайте последствията от всеки избор
   - Правете въпросите прости и ясни

3. **Обработвайте непредвиден вход**
   - Валидирайте отговорите на потребителя
   - Осигурете стойности по подразбиране при невалиден вход
   - Давайте ясни съобщения за грешки

4. **Проследявайте request_id**
   - Използвайте корелацията между request_id и отговори
   - Не се опитвайте да управлявате състоянието ръчно

5. **Проектирайте за неблокиране**
   - Не блокирайте нишки, чакайки вход
   - Използвайте async шаблони навсякъде
   - Поддържайте едновременни инстанции на работния процес

### 📚 Свързани концепции:

- **Agent Middleware** - Засичане на обаждания към агенти (предишната тетрадка)
- **Управление на състоянието на работния процес** - Запазване на състоянието между изпълнения
- **Мулти-агентно сътрудничество** - Комбиниране на човек в цикъла с екипи от агенти
- **Архитектура, базирана на събития** - Създаване на реактивни системи с помощта на събития

---

### 🎓 Поздравления!

Вие овладяхте работни процеси с човек в цикъла в Microsoft Agent Framework! Сега знаете как да:
- ✅ Поставяте работни процеси на пауза за събиране на човешки вход
- ✅ Използвате RequestInfoExecutor и RequestInfoMessage
- ✅ Обработвате изпълнение на потоци със събития
- ✅ Създавате персонализирани изпълнители с @handler
- ✅ Насочвате работни процеси въз основа на човешки решения
- ✅ Създавате интерактивни AI агенти, които си сътрудничат с хора

**Това е критичен шаблон за изграждане на надеждни, контролируеми AI системи!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
